In [1]:
import torch
from torchvision import transforms
from torchvision.datasets import FashionMNIST

train_dataset = FashionMNIST(root="./data/train", train=True, download=False, transform=transforms.ToTensor())
test_dataset = FashionMNIST(root="./data/test", train=False, download=False, transform=transforms.ToTensor())

print(train_dataset, test_dataset, sep="\n\n", end="\n\n")
print(f"shape of images → {train_dataset[0][0].shape}")
print(f"class labels → {train_dataset.classes}")

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.get_device_name()

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data/train
    Split: Train
    StandardTransform
Transform: ToTensor()

Dataset FashionMNIST
    Number of datapoints: 10000
    Root location: ./data/test
    Split: Test
    StandardTransform
Transform: ToTensor()

shape of images → torch.Size([1, 28, 28])
class labels → ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


'NVIDIA GeForce MX350'

In [ ]:
from torch.utils.data import Dataset, DataLoader
class FashionMNIST(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index][0], self.data[index][1]

train_df = FashionMNIST(data=train_dataset)
test_df = FashionMNIST(data=test_dataset)

In [15]:
# neural network architecture
from torch import nn
from torch.nn import Module

class CustomModel(Module):
    def __init__(self, activation, in_features:int=1*28*28, out_features:int=10, num_hidden_layers:int=2, num_neurons:int=16, decay:float=0.5, dropout_rate:float=0.2):
        super().__init__()

        self.in_features = in_features
        self.num_hidden_layers = num_hidden_layers
        self.num_neurons = num_neurons
        self.activation = activation

        match self.activation:
            case "leaky_rely":
                self.activation = nn.LeakyReLU()
            
            case "p_relu":
                self.activation = nn.PReLU()

            case "elu":
                self.activation = nn.ELU()
            
            case "gelu":
                self.activation = nn.GELU()
            
            case "mish":
                self.activation = nn.Mish()
            
            case _:
                self.activation = nn.ReLU()

        model_layers = []
        flatten_layer = nn.Flatten(start_dim=1, end_dim=-1) # from num_rows to num_cols and skips num_channels dimension
        model_layers.append(flatten_layer)

        for _ in range(self.num_hidden_layers):
            layer = nn.Linear(in_features=self.in_features, out_features=self.num_neurons)
            batch_norm = nn.BatchNorm1d(num_features=self.num_neurons)
            activation = self.activation
            dropout = nn.Dropout1d(p=dropout_rate)


            model_layers.extend([layer, batch_norm, activation, dropout])
            self.in_features = self.num_neurons
            self.num_neurons = int(self.num_neurons * decay)

            if self.num_neurons <= 10: break
        
        output_layer = nn.Linear(in_features=self.in_features, out_features=out_features)
        model_layers.append(output_layer)

        self.model = nn.Sequential(*model_layers) # dynamic sequential model
        
    def _initialize_weights(self):
        """ Initialize weights using He initialization for ReLU networks """
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')

                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

    def forward(self, features):
        return self.model(features)

In [16]:
def train(epochs, train_loader, model, criterion, optimizer) -> CustomModel:
    """ training pipeline """
    model.train()

    for _ in range(epochs):

        for batch_features, batch_labels in train_loader:
            
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            
            loss = criterion(outputs, batch_labels) # explicitely add regularization terms to apply regularization
            loss.backward()

            optimizer.zero_grad()
            optimizer.step()

    return model

In [17]:
from torchmetrics.classification import MulticlassF1Score
def evaluate(model, test_loader):
    """ evaluation pipeline """
    model.eval()
    f1 = MulticlassF1Score(num_classes=10, average='macro', multidim_average="global").to(device)

    with torch.no_grad():

        for batch_features, batch_labels in test_loader:

            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            f1.update(outputs, batch_labels)
    
    test_f1 = f1.compute()
    return test_f1

In [18]:
# objective function
def objective(trial):
    
    # choice of hyperparameter valued from the search space for next trail
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)
    neurons_decay_rate = trial.suggest_float("neurons_decay_rate", 0.3, 0.8, step=0.1)
    activation = trial.suggest_categorical("activation_function", ["leaky_relu", "p_relu", "elu", "gelu", "mish"])
    epochs = trial.suggest_int("epochs", 10, 50, step=10)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    train_loader = DataLoader(
        dataset=train_df,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    test_loader = DataLoader(
        dataset=test_df,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    model = CustomModel(in_features=1*28*28, out_features=10, num_hidden_layers=num_hidden_layers, num_neurons=neurons_per_layer, decay=neurons_decay_rate, dropout_rate=dropout_rate, activation=activation).to(device=device)

    criterion = nn.CrossEntropyLoss()

    match optimizer_name:
        case "Adam":
            optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

        case "SGD":
            optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

        case _:
            optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    model = train(epochs=epochs, train_loader=train_loader, model=model, criterion=criterion, optimizer=optimizer)
    f1_score = evaluate(model=model, test_loader=test_loader)

    return float(f1_score)

In [19]:
import optuna

study = optuna.create_study(direction='maximize') # study → experiment
study.optimize(objective, n_trials=10) # 10 trials in one experiment

[I 2026-08-18 07:00:38,144] A new study created in memory with name: no-name-ae703846-86b2-4177-b23b-6da7066886c7
[I 2026-08-18 07:12:40,905] Trial 0 finished with value: 0.07580549269914627 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 72, 'neurons_decay_rate': 0.6000000000000001, 'activation_function': 'leaky_relu', 'epochs': 50, 'learning_rate': 2.459338407750373e-05, 'dropout_rate': 0.2, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 0.0002516491414003685}. Best is trial 0 with value: 0.07580549269914627.
[I 2026-08-18 07:14:20,334] Trial 1 finished with value: 0.060717396438121796 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 8, 'neurons_decay_rate': 0.5, 'activation_function': 'elu', 'epochs': 10, 'learning_rate': 0.03636726880641448, 'dropout_rate': 0.30000000000000004, 'batch_size': 128, 'optimizer': 'SGD', 'weight_decay': 3.1382271760815415e-05}. Best is trial 0 with value: 0.07580549269914627.
[I 2026-08-18 07:32:21,061] Trial 2 finished

In [ ]:
print(study.best_trial)
print(study.best_params)
print(study.best_value) # increase number of trials to get greater scores

FrozenTrial(number=2, state=<TrialState.COMPLETE: 1>, values=[0.08772982656955719], datetime_start=datetime.datetime(2026, 8, 18, 7, 14, 20, 335126), datetime_complete=datetime.datetime(2026, 8, 18, 7, 32, 21, 61464), params={'num_hidden_layers': 1, 'neurons_per_layer': 40, 'neurons_decay_rate': 0.4, 'activation_function': 'elu', 'epochs': 50, 'learning_rate': 0.05678075249846114, 'dropout_rate': 0.4, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 2.6854171357379313e-05}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'num_hidden_layers': IntDistribution(high=5, log=False, low=1, step=1), 'neurons_per_layer': IntDistribution(high=128, log=False, low=8, step=8), 'neurons_decay_rate': FloatDistribution(high=0.8, log=False, low=0.3, step=0.1), 'activation_function': CategoricalDistribution(choices=('leaky_relu', 'p_relu', 'elu', 'gelu', 'mish')), 'epochs': IntDistribution(high=50, log=False, low=10, step=10), 'learning_rate': FloatDistribution(high=0.1, 